# Neo4J Friends Graph

References: https://neo4j.com/developer/

In [ ]:
#!pip install neo4j networkx --upgrade

In [ ]:
from neo4j import GraphDatabase, basic_auth

import networkx as nx
import matplotlib.pyplot as plt

## Neo4J Connection

In [ ]:
url = "bolt://localhost:7687"
user = "neo4j"
password = "Password"

driver = GraphDatabase.driver(url, auth=basic_auth(user, password))
session = driver.session()
session

In [ ]:
# Create a session and try a simple query to check the connection
try:
    with driver.session() as session:
        # Run a simple query to test the connection
        result = session.run("RETURN 'Connection is working!' AS message")
        
        # Fetch the result
        message = result.single()["message"]
        print(message)
        
except Exception as e:
    print(f"Could not connect to Neo4j: {e}")
finally:
    # Close the driver to free up resources
    driver.close()

## Create Friends Graph

In [ ]:
# Create a driver instance
driver = GraphDatabase.driver(url, auth=(user, password))

# Insert data query

# UNWIND expands a list into a sequence of rows. 
insert_query = '''
UNWIND $pairs as pair
MERGE (p1:Person {name: pair[0]})
MERGE (p2:Person {name: pair[1]})
MERGE (p1)-[:KNOWS]-(p2);
'''

# Data to insert
data = [
    ["Jim", "Mike"],
    ["Jim", "Billy"],
    ["Anna", "Jim"],
    ["Anna", "Mike"],
    ["Sally", "Anna"],
    ["Joe", "Sally"],
    ["Joe", "Bob"],
    ["Bob", "Sally"]
]

# Execute the query
with driver.session() as session:
    session.run(insert_query, parameters={"pairs": data})


### Visualize Graph

In [ ]:
# Query the graph (adjust query based on your graph structure)
query = """
MATCH (p:Person)-[:KNOWS]->(friend:Person)
RETURN p.name AS person, collect(friend.name) AS friends
"""
with driver.session() as session:
    data = session.run(query).data()

# Create a NetworkX graph and add nodes and edges
G = nx.Graph()

for record in data:
    person = record["person"]
    friends = record["friends"]
    for friend in friends:
        G.add_edge(person, friend)
        #print(person)        
    
# Draw the graph
nx.draw(G, with_labels=True, node_color="skyblue", node_size=2000, font_size=10, font_weight="bold")
plt.show()

## Graph Queries

### Friend of a Friend (FOAF) Pattern

In [ ]:
# Friends of a friend (FOAF)
foaf_query = '''
MATCH (person:Person)-[:KNOWS]-(friend)-[:KNOWS]-(foaf) 
WHERE person.name = $name
  AND NOT (person)-[:KNOWS]-(foaf)
RETURN foaf.name AS name
'''

# The name of the person for whom we want to find Friends of a Friend
person_name = "Joe" 

# Run the query and get the result
with driver.session() as session:
    results = session.run(foaf_query, name=person_name).data()

# Create a graph using networkx
G = nx.Graph()

# Add the main person to the graph
G.add_node(person_name, color='blue')

# Add the friends of a friend to the graph
for record in results:
    foaf_name = record["name"]
    print(foaf_name)

### Mutual Friends Pattern

In [ ]:
# Common friends
common_friends_query = """
MATCH (user:Person)-[:KNOWS]-(friend)-[:KNOWS]-(foaf:Person)
WHERE user.name = $user AND foaf.name = $foaf
RETURN friend.name AS friend
"""

# Run the query and get the result
with driver.session() as session:
    results = session.run(common_friends_query, parameters={"user": "Jim", "foaf": "Mike"}).data()

# Draw the graph
G = nx.Graph()

# Add nodes for Jim and Mike (the people we're interested in)
G.add_node("Jim", color="blue")
G.add_node("Mike", color="blue")

# Add the common friends (edges between Jim and Mike through the common friends)
for record in results:
    friend_name = record["friend"]
    G.add_node(friend_name, color="green")  # Green color for the common friends
    G.add_edge("Jim", friend_name)
    G.add_edge("Mike", friend_name)

# Draw the graph using NetworkX and Matplotlib
node_colors = [node[1]['color'] for node in G.nodes(data=True)]  # Color nodes based on type

plt.figure(figsize=(4, 4))
nx.draw(G, with_labels=True, node_size=3000, node_color=node_colors, font_size=12, font_weight="bold", edge_color='gray')
plt.title("Common Friends of Jim and Mike")
plt.show()


### Shortest Path

In [ ]:
# Query to find the shortest path between two nodes
connecting_paths_query = """
MATCH path = shortestPath((p1:Person)-[:KNOWS*..6]-(p2:Person))
WHERE p1.name = $name1 AND p2.name = $name2
RETURN path
"""

# Run the query and get the result
with driver.session() as session:
    results = session.run(connecting_paths_query, parameters={"name1": "Joe", "name2": "Billy"}).data()

# Draw the graph
G = nx.Graph()

# Process the path results
for record in results:
    path = record["path"]
    print(path)

In [ ]:
# Close the session
session.close()